# Phase 2B — Dimensionality Reduction (real 20K Qwen embeddings)

Aligned with instructor `starter_code/phase2_matrix_ops/svd_power_iteration.py`. Compare custom **block power-iteration SVD**, sklearn **TruncatedSVD**, sklearn **PCA**; NumPy full SVD as a **small-scale ground-truth reference**.

**Do not confuse fitting time with search time.** We measure fitting/reduction time and reconstruction error. Search speed after reduction needs its own benchmark.

Start with **PILOT=True**. After verifying, set **PILOT=False** for full measurements. No synthetic matrices or invented results.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, gc, json
from pathlib import Path
from sklearn.decomposition import TruncatedSVD, PCA
from google.colab import drive

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
OUT=ROOT/'phase2_matrix_ops'
DATA=OUT/'2A_sparse_dense'/'qwen_20k_minilm_384_float32.npy'
assert DATA.exists(), f'Cached Phase 2A embedding not found: {DATA}'
X=np.load(DATA, mmap_mode='r')
assert X.shape==(20000,384), X.shape
assert X.dtype==np.float32, X.dtype
print('Real embeddings:',X.shape, X.dtype, 'source:',DATA)
(OUT/'2B_dimensionality_reduction').mkdir(parents=True,exist_ok=True)
(OUT/'benchmarks').mkdir(parents=True,exist_ok=True)
(OUT/'figures').mkdir(parents=True,exist_ok=True)

## Our own block power iteration SVD

Start with random orthonormal right directions `V`, repeatedly multiply by `X.T @ (X @ V)`, orthonormalize using QR, and stop when the **subspace** stabilizes. A final small SVD of `X @ V` gives singular values and orthogonal left/right vectors. Using a subspace convergence test rather than `||V_new-V||` avoids false nonconvergence caused by sign flips or rotations. Matrix multiplication is allowed: the assignment asks us to implement the *iterative algorithm*, not manually code BLAS.

In [ ]:
def power_iteration_svd(X, k, max_iter=35, tol=1e-4, seed=42):
    """Custom top-k block power iteration; return U, S, Vt and iteration count."""
    X=np.asarray(X, dtype=np.float64)
    m,n=X.shape
    if not 1<=k<=min(m,n): raise ValueError('invalid k')
    rng=np.random.default_rng(seed)
    V,_=np.linalg.qr(rng.standard_normal((n,k)), mode='reduced')
    iterations=0
    for it in range(max_iter):
        W=X.T@(X@V)
        Vnew,_=np.linalg.qr(W, mode='reduced')
        # Principal-angle/subspace residual, invariant to sign or basis rotation
        residual=np.linalg.norm(Vnew-V@(V.T@Vnew),ord='fro')/np.sqrt(k)
        V=Vnew
        iterations=it+1
        if residual<tol: break
    B=X@V
    Ub,S,Rt=np.linalg.svd(B,full_matrices=False)
    Vt=Rt@V.T
    return Ub[:,:k],S[:k],Vt[:k,:],iterations

def relative_reconstruction_error(X, U, S, Vt):
    """||X - U diag(S) Vt||_F / ||X||_F; report also absolute error."""
    A=np.asarray(X,dtype=np.float64)
    reconstructed=(U*S[None,:])@Vt
    absolute=np.linalg.norm(A-reconstructed,'fro')
    return float(absolute),float(absolute/np.linalg.norm(A,'fro'))

# Correctness sanity checks on real data; no synthetic matrices
A=np.asarray(X[:300],dtype=np.float64)
U,S,Vt,it=power_iteration_svd(A,10,max_iter=50)
assert U.shape==(300,10) and S.shape==(10,) and Vt.shape==(10,384)
assert np.all(np.isfinite(S)) and np.all(np.diff(S)<=1e-7)
assert np.allclose(U.T@U,np.eye(10),atol=1e-5)
assert np.allclose(Vt@Vt.T,np.eye(10),atol=1e-5)
_,s_ref,_=np.linalg.svd(A,full_matrices=False)
print('Power iteration sanity passed; iterations:',it)
print('Leading singular values ours/reference:',S[:3],s_ref[:3])

## Pilot and full benchmark settings

Pilot: k=50 at 1K documents, three methods; full SVD reference at 1K. Full: k=50/100/200 and sizes 1K/5K/10K/20K. **Each fit includes the method's reduction training cost**. Reconstruction error is on the same input matrix for each method; PCA uses its mean when reconstructing. For TruncatedSVD and PCA, reconstruction is computed by `inverse_transform`. Matrix storage uses float32 projected data for comparability; fitted model and temporary allocations are excluded. 

Full SVD is cubic/expensive; we only run it on the 1K subset as a reference. Power iteration can be slow at k=200 on CPU; monitor runtime.

In [ ]:
PILOT=True  # Change to False only after the pilot succeeds
K_VALUES=[50] if PILOT else [50,100,200]
SIZES=[1000] if PILOT else [1000,5000,10000,20000]
METHODS=['power_iteration','truncated_svd','pca']

def run_one(A,method,k):
    start=time.perf_counter()
    if method=='power_iteration':
        U,S,Vt,iterations=power_iteration_svd(A,k,max_iter=35,tol=1e-4)
        Z=(U*S[None,:]).astype(np.float32)
        Ahat=(U*S[None,:])@Vt
    elif method=='truncated_svd':
        model=TruncatedSVD(n_components=k,n_iter=7,random_state=42,algorithm='randomized')
        Z=model.fit_transform(A).astype(np.float32)
        Ahat=model.inverse_transform(Z)
        iterations=np.nan
    elif method=='pca':
        model=PCA(n_components=k,svd_solver='randomized',random_state=42,iterated_power=7)
        Z=model.fit_transform(A).astype(np.float32)
        Ahat=model.inverse_transform(Z)
        iterations=np.nan
    else: raise ValueError(method)
    fit_and_reconstruct_s=time.perf_counter()-start
    absolute=float(np.linalg.norm(np.asarray(A,dtype=np.float64)-Ahat,'fro'))
    relative=absolute/float(np.linalg.norm(A,'fro'))
    return dict(fit_reconstruct_s=fit_and_reconstruct_s,fro_error=absolute,relative_fro_error=relative,
                projected_mib=Z.nbytes/2**20,iterations=iterations)

records=[]
for n in SIZES:
    A=np.array(X[:n],dtype=np.float32,copy=True)
    for k in K_VALUES:
        for method in METHODS:
            print('Running',n,k,method,flush=True)
            rec=run_one(A,method,k)
            records.append(dict(n_docs=n,k=k,method=method,**rec))
            print(' ', 'seconds:',round(rec['fit_reconstruct_s'],3),'relative error:',round(rec['relative_fro_error'],5),flush=True)
    del A; gc.collect()
results=pd.DataFrame(records)
print(results.to_string(index=False))
suffix='pilot' if PILOT else 'full'
results.to_csv(OUT/'benchmarks'/f'phase2b_methods_{suffix}.csv',index=False)
print('Saved:',OUT/'benchmarks'/f'phase2b_methods_{suffix}.csv')

## NumPy full SVD ground truth on a small real subset

Full SVD has the optimal rank-k reconstruction for the *uncentered* matrix. PCA first centers data, so its reconstruction objective differs. We intentionally do **not** perform full SVD on all 20K rows in this CPU notebook.

In [ ]:
A=np.asarray(X[:1000],dtype=np.float64)
t0=time.perf_counter()
U0,S0,Vt0=np.linalg.svd(A,full_matrices=False)
full_svd_time=time.perf_counter()-t0
reference=[]
for k in [50,100,200]:
    abs_err,rel_err=relative_reconstruction_error(A,U0[:,:k],S0[:k],Vt0[:k,:])
    reference.append(dict(n_docs=1000,k=k,method='numpy_full_svd_reference',svd_fit_s=full_svd_time,fro_error=abs_err,relative_fro_error=rel_err))
ref_df=pd.DataFrame(reference)
print(ref_df.to_string(index=False))
ref_df.to_csv(OUT/'benchmarks'/'phase2b_full_svd_reference_1k.csv',index=False)

## Figures: time vs k, time vs N, reconstruction error and memory

The pilot produces only one point per curve, so full scaling plots are meaningful **only after** setting `PILOT=False`. No claim of a winner until the full run.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(13,9))
for method,g in results.groupby('method'):
    g20=g[g.n_docs==max(SIZES)].sort_values('k')
    g50=g[g.k==min(K_VALUES)].sort_values('n_docs')
    axes[0,0].plot(g20.k,g20.fit_reconstruct_s,'o-',label=method)
    axes[0,1].plot(g50.n_docs,g50.fit_reconstruct_s,'o-',label=method)
    axes[1,0].plot(g20.k,g20.relative_fro_error,'o-',label=method)
    axes[1,1].plot(g50.n_docs,g50.projected_mib,'o-',label=method)
axes[0,0].set(title=f'Fit + reconstruction time vs k (N={max(SIZES)})',xlabel='Components k',ylabel='Seconds')
axes[0,1].set(title=f'Time vs corpus size (k={min(K_VALUES)})',xlabel='Documents N',ylabel='Seconds')
axes[1,0].set(title=f'Relative Frobenius error (N={max(SIZES)})',xlabel='Components k',ylabel='Relative error')
axes[1,1].set(title=f'Projected float32 matrix storage (k={min(K_VALUES)})',xlabel='Documents N',ylabel='MiB')
for ax in axes.flat: ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.tight_layout()
p=OUT/'figures'/f'phase2b_comparison_{suffix}.png'
fig.savefig(p,dpi=150,bbox_inches='tight');plt.show();print('Saved',p)

# Theoretical projected matrix size: no claim about whole process RAM
mem=pd.DataFrame([{'n_docs':n,'k':k,'original_384_float32_mib':n*384*4/2**20,
                   'projected_float32_mib':n*k*4/2**20,'storage_reduction_pct':100*(1-k/384)}
                  for n in [1000,5000,10000,20000] for k in [50,100,200]])
mem.to_csv(OUT/'benchmarks'/'phase2b_projected_matrix_memory.csv',index=False)
print(mem[mem.n_docs==20000].to_string(index=False))

## Interpretation checklist (fill only after full measurements)

1. Which method reaches low reconstruction error for k=50,100,200?
2. How do fitting times change with k and N?
3. Why is truncated SVD often faster than full SVD? It computes only the leading components instead of a complete decomposition; exact crossover is hardware/data dependent and **must be measured**.
4. Why PCA differs: it centers columns, whereas TruncatedSVD normally does not.
5. What is the largest tested k where TruncatedSVD is faster? Use only the measured tested range; no extrapolation.
6. **Search latency and retrieval quality** are not established by reconstruction error alone; benchmark reduced-space search and top-k overlap before choosing final k.

Assignment source: [instructor starter](https://github.com/adaskin/hpc-for-ai-course/blob/main/projects/project1-hp-ss-engine/starter_code/phase2_matrix_ops/svd_power_iteration.py).